This code is just a preliminary look at some of the clarity data.

You will need to download the ~2GB dataset from: https://data.cityofchicago.org/Health-Human-Services/Open-Air-Chicago-Individual-Measurements/xfya-dxtq/about_data
In the top right, click on Export and download the file as a .csv (note: This export process does take some time, so be patient).


Notebook authorship
Author: Noah Ring
Year: 2026
Program: EarthRISE Developers Academy

In [1]:
#Some of these packages are unused.
import pandas as pd
import datetime as dt
from datetime import datetime
from dateutil.relativedelta import relativedelta
import xarray as xr
import matplotlib.pyplot as plt
from mpl_toolkits.basemap import Basemap
import seaborn as sb
import geopandas as gpd
import geodatasets as geodat
import numpy as np
import seaborn as sns
from matplotlib_scalebar.scalebar import ScaleBar
import matplotlib.colors as mcolors


colors = ['#dadaeb','#bcbddc',
          '#9e9ac8',
          '#807dba','#6a51a3','#54278f','#3f007d']

cmap = mcolors.LinearSegmentedColormap.from_list('purples_custom', colors)

#Change the matplotlib dpi to increase or decrease output size in vscode
plt.rcParams['figure.dpi'] = 200 

#Define the path where your data is stored.
data_folder = "~/Documents/01_Research/larc-chicago-haq/datasets/"

#Load chicago community population dataset. We will pull shapefiles from this.
#This just has population from 2010, but honestly it would potentially be interesting to do no2 levels per 100k or something 
#i.e. alludes to exposure
chicago = gpd.read_file(geodat.get_path("geoda.chicago_commpop"))

#look at the first five rows
chicago.head()


/home/nlring/.conda/envs/chi-haq/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


,community,NID,POP2010,POP2000,POPCH,POPPERCH,popplus,popneg,geometry
0,DOUGLAS,35,18238,26470,-8232,-31.099358,0,1,"MULTIPOLYGON (((-87.60914 41.84469, -87.60915 ..."
1,OAKLAND,36,5918,6110,-192,-3.142390,0,1,"MULTIPOLYGON (((-87.59215 41.81693, -87.59231 ..."
2,FULLER PARK,37,2876,3420,-544,-15.906433,0,1,"MULTIPOLYGON (((-87.6288 41.80189, -87.62879 4..."
3,GRAND BOULEVARD,38,21929,28006,-6077,-21.698922,0,1,"MULTIPOLYGON (((-87.60671 41.81681, -87.6067 4..."
4,KENWOOD,39,17841,18363,-522,-2.842673,0,1,"MULTIPOLYGON (((-87.59215 41.81693, -87.59215 ..."


In [2]:

#Define filename for dataset here.
file_dir = data_folder + "Open_Air_Chicago_Individual_Measurements_20260615.csv"

#Read in the full dataset for clarity sensors
cook_full = pd.read_csv(file_dir)


In [ ]:
cook_clarity = cook_full[['time', 'datasourceid', 'sensor_name', 'latitude', 'longitude', 'no2ConcIndividual.value',
                            'pm2_5ConcNumIndividual.value', 'pm2_5ConcNumIndividual.raw']]

cook_clarity["time"] = pd.to_datetime(cook_clarity["time"], format="%m/%d/%Y %I:%M:%S %p")
cook_clarity["month"] = cook_clarity["time"].dt.month
cook_clarity["year"] = cook_clarity["time"].dt.year
cook_clarity["month_year_str"] = cook_clarity['month'].astype(str) + "-" + cook_clarity['year'].astype(str)
cook_clarity["month_year"] = pd.to_datetime(cook_clarity["month_year_str"], format="%m-%Y")
# datetime.strptime(cook_clarity["time"], "%m/%d/%Y %I:%M:%S %p")


#Convert to geopandas dataframe
cook_clarity["geometry"] = gpd.points_from_xy(
    x = cook_clarity["longitude"],
    y = cook_clarity["latitude"]
)

#Set coordinate reference system to match crs of data we downloaded earlier
cook_clarity = gpd.GeoDataFrame(cook_clarity).set_crs(chicago.crs)

#Print type
print(type(cook_clarity))

#This line just gets the sensor count. 
print(np.unique(cook_clarity['sensor_name']).shape[0])
print(np.unique(cook_clarity['latitude']).shape[0])

#Print the first five lines
print(cook_clarity.head(5))




<class 'geopandas.geodataframe.GeoDataFrame'>
286
280
                 time               sensor_name   latitude  longitude  \
0 2026-01-18 13:14:59            Logan Square 3  41.921811 -87.690637   
1 2026-01-18 12:24:16             Irving Park 4  41.941523 -87.735605   
2 2026-06-10 23:05:39               West Town 4  41.894832 -87.672764   
3 2026-03-27 09:36:07               Englewood 1  41.789686 -87.652596   
4 2026-04-28 07:23:59  Greater Grand Crossing 6  41.751465 -87.636296   

   no2ConcIndividual.value  pm2_5ConcNumIndividual.value  \
0                      NaN                          3.01   
1                      NaN                          1.58   
2                    17.55                          9.49   
3                     9.21                          3.92   
4                     8.69                         18.79   

   pm2_5ConcNumIndividual.raw  month  year month_year_str month_year  \
0                        3.01      1  2026         1-2026 2026-01-01   
1 

KeyError: "Could not find 'datasource id' or 'datasource_id' in cook_full columns."

In [4]:


unique_sources = (
    cook_full[["sensor_name","datasourceid"]]
    .dropna(subset=["sensor_name",'datasourceid'])
    .drop_duplicates()
    .sort_values(by=["sensor_name"])
)

output_csv = "cook_full_unique_datasource_sensor.csv"
unique_sources.to_csv(output_csv, index=False)
print(f"Wrote {len(unique_sources)} unique rows to {output_csv}")

Wrote 286 unique rows to cook_full_unique_datasource_sensor.csv


In [5]:
#get % of missing per sensor
#note that this is just for the "no2ConcIndividual.value" variable in the dataframe
cook_clarity_na_count = (
    cook_clarity.groupby(["sensor_name", "longitude", "latitude","month", "month_year"])["no2ConcIndividual.value"]
    .apply(lambda x: 100 * x.isna().mean())
    .reset_index()
)


cook_clarity_na_count = cook_clarity_na_count.merge(
    cook_clarity.groupby(["sensor_name", "longitude", "latitude","month","month_year"])
    .size()
    .reset_index(name="num_vals"),
    on=["sensor_name", "longitude", "latitude", "month","month_year"],
    how="left",
)


cook_clarity_na_count.rename(columns={"no2ConcIndividual.value": "percent_missing_no2"}, inplace=True)

# cook_clarity_na_count.to_csv("~/Documents/01_Research/larc-chicago-haq/datasets/missing-output.csv")

#Print first five rows
print(cook_clarity_na_count.head())

#Convert above dataframe to geopandas dataframe
#important to assign longitude and latitude for plotting
#This will need to be done for other statistics we calculate per station as well.
cook_clarity_na_count["geometry"] = gpd.points_from_xy(
    x=cook_clarity_na_count["longitude"],
    y=cook_clarity_na_count["latitude"]
)

#Set coordinate reference system to match crs of data we downloaded earlier
cook_clarity_na_count = gpd.GeoDataFrame(cook_clarity_na_count).set_crs(chicago.crs)

# Filter cook_full to requested sensor names.
name_pattern = r"Mayfair|Schiller Park|Washington HS"
cook_full_target_sensors = cook_full[
    cook_full["sensor_name"].str.contains(name_pattern, case=False, na=False)
].copy()

print(cook_full_target_sensors[["sensor_name"]].drop_duplicates().sort_values("sensor_name"))
print(f"Filtered rows: {len(cook_full_target_sensors)}")

     sensor_name  longitude   latitude  month month_year  percent_missing_no2  \
0  Albany Park 1 -87.725577  41.975287      1 2026-01-01            70.611133   
1  Albany Park 1 -87.725577  41.975287      2 2026-02-01            50.625270   
2  Albany Park 1 -87.725577  41.975287      3 2026-03-01            15.066354   
3  Albany Park 1 -87.725577  41.975287      4 2026-04-01             0.403551   
4  Albany Park 1 -87.725577  41.975287      5 2026-05-01             0.000000   

   num_vals  
0      2569  
1      2319  
2      2562  
3      2478  
4      4691  
                     sensor_name
414        Mayfair Collocation 1
71         Mayfair Collocation 2
327        Mayfair Collocation 3
25   Schiller Park Collocation 1
83   Schiller Park Collocation 2
297  Schiller Park Collocation 3
269  Washington HS Collocation 1
168  Washington HS Collocation 2
306  Washington HS Collocation 3
Filtered rows: 213695


In [6]:
cook_full_target_sensors

,datasourceid,time,sensor_name,atmPressureIndividual.raw,atmPressureIndividual.value,bcAllSourcesIndividual.raw,bcAllSourcesIndividual.value,bcBiomassIndividual.raw,bcBiomassIndividual.value,bcFossilFuelIndividual.raw,...,temperatureInternalIndividual.raw,temperatureInternalIndividual.value,windDirectionIndividual.raw,windDirectionIndividual.value,windSpeedIndividual.raw,windSpeedIndividual.value,latitude,longitude,location,record_id
25,DWKUZ0134,01/23/2026 01:06:41 PM,Schiller Park Collocation 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-21.93,-21.93,NaN,NaN,NaN,NaN,41.965175,-87.876252,POINT (-87.876252 41.965175),DWKUZ0134_2026-01-23T13_06
71,DVYCN4987,03/27/2026 06:15:32 PM,Mayfair Collocation 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,6.09,6.09,NaN,NaN,NaN,NaN,41.964742,-87.749434,POINT (-87.749434 41.964742),DVYCN4987_2026-03-27T18_15
83,DLAIF4030,04/01/2026 07:24:59 PM,Schiller Park Collocation 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,3.86,3.86,NaN,NaN,NaN,NaN,41.965175,-87.876252,POINT (-87.876252 41.965175),DLAIF4030_2026-04-01T19_24
168,DDFTD2975,06/10/2026 11:28:27 PM,Washington HS Collocation 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,22.63,22.63,NaN,NaN,NaN,NaN,41.687495,-87.538083,POINT (-87.538083 41.687495),DDFTD2975_2026-06-10T23_28
206,DDFTD2975,04/02/2026 10:50:27 PM,Washington HS Collocation 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,20.90,20.90,NaN,NaN,NaN,NaN,41.687495,-87.538083,POINT (-87.538083 41.687495),DDFTD2975_2026-04-02T22_50
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8167989,DIPDE7442,03/27/2026 06:10:23 PM,Schiller Park Collocation 3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.45,5.45,NaN,NaN,NaN,NaN,41.965175,-87.876252,POINT (-87.876252 41.965175),DIPDE7442_2026-03-27T18_10
8167990,DEQFM3089,01/23/2026 12:58:11 PM,Mayfair Collocation 3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,-21.65,-21.65,NaN,NaN,NaN,NaN,41.964742,-87.749434,POINT (-87.749434 41.964742),DEQFM3089_2026-01-23T12_58
8168058,DFNHF0256,03/28/2026 12:00:02 AM,Mayfair Collocation 1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,2.71,2.71,NaN,NaN,NaN,NaN,41.964742,-87.749434,POINT (-87.749434 41.964742),DFNHF0256_2026-03-28T00_00
8168066,DDFTD2975,03/22/2026 04:05:10 PM,Washington HS Collocation 2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,5.71,5.71,NaN,NaN,NaN,NaN,41.687495,-87.538083,POINT (-87.538083 41.687495),DDFTD2975_2026-03-22T16_05


In [ ]:
chi_result_na = cook_clarity_na_count

months = chi_result_na['month_year'].unique()
months = months[months.argsort()]

start_date = chi_result_na['month_year'].min()
end_date = chi_result_na['month_year'].max()

delta = relativedelta(months=1)

fig, axes = plt.subplots(
    nrows=3, ncols=4,
    figsize=(20, 15)
)

for ax, month in zip(axes.flatten(), range(1,13)):
    # ax = axes[i]
    
    # Base Chicago boundary
    chicago.plot(color='white', edgecolor='black', ax=ax)
    
    # Filter to month
    monthly_data = chi_result_na[chi_result_na['month_year'] == start_date]
    
    if not monthly_data.empty:
        monthly_data.plot(ax=ax, column='percent_missing_no2', cmap='Bu',
                          vmin=chi_result_na['percent_missing_no2'].min(),
                          vmax=chi_result_na['percent_missing_no2'].max())
    
    # ax.set_title(calendar.month_name[month])
    # ax.axis('off')
  
    title_date = dt.date(start_date.year, start_date.month, start_date.day).strftime('%B %Y')
    ax.set_title(title_date, fontsize=12)
    ax.axis('off')
    start_date += delta

# Shared colorbar
sm = plt.cm.ScalarMappable(
    cmap='BuPu',
    norm=plt.Normalize(
        vmin=chi_result_na['percent_missing_no2'].min(),
        vmax=chi_result_na['percent_missing_no2'].max()
    )
)
sm._A = []
fig.colorbar(sm, ax=axes, shrink=0.5, label='%')

plt.suptitle('% Data Missing - Chicago', fontsize=16, y=1.02)
plt.tight_layout()
# plt.savefig("month_percent_missing-individual.png")
plt.show()

In [ ]:

#Print first five rows
print(cook_clarity_na_count.head())

#Convert above dataframe to geopandas dataframe
#important to assign longitude and latitude for plotting
#This will need to be done for other statistics we calculate per station as well.
cook_clarity_na_count["geometry"] = gpd.points_from_xy(
    x=cook_clarity_na_count["longitude"],
    y=cook_clarity_na_count["latitude"]
)

#Set coordinate reference system to match crs of data we downloaded earlier
cook_clarity_na_count = gpd.GeoDataFrame(cook_clarity_na_count).set_crs(chicago.crs)


In [ ]:

#we can peak at the first five lines of the geodataframe
print(cook_clarity_na_count.head())

#lets print to make sure its in the right format
print(type(cook_clarity_na_count))

#Print the CRS of both geodataframes to confirm
print(chicago.crs)
print(cook_clarity_na_count.crs)

#Now we construct a first map, where the color corresponds to the number of missing values per sensor.
ax = chicago.plot(color='white', edgecolor='black')

cook_clarity_na_count.plot(
    ax=ax,
    # markersize=0.05*cook_clarity_na_count['num_missing_no2']**2, 
    markersize=cook_clarity_na_count['percent_missing_no2'],
    figsize=(10,10),
    column='percent_missing_no2',
    cmap=cmap, 
    legend=True)

# ax.add_artist(ScaleBar(1))
plt.title("% of Missing NO2 Measurements")
plt.savefig("perc_missing_stations.png")

In [ ]:
#now we combine the clarity point data and datasets

chi_joined = gpd.sjoin(cook_clarity, 
                        chicago[["community", "NID", "geometry"]],
                        how="left",
                        predicate="within")

# calculate the number of points per community
counts = (
    chi_joined.groupby(["community", "NID"])
    .size()
    .reset_index(name="point_count")
)

#calculate the average NO2 value per community
chi_summary = (
    chi_joined.groupby(["community", "NID"])
    .agg(
        avg_value=("no2ConcIndividual.value", "mean")
    )
    .reset_index()
)

#left join the datasets to the chicago community shapefile data
chi_result = chicago.merge(chi_summary, on="NID", how="left")
chi_result = chi_result.merge(counts, on="NID", how="left")

chi_result

In [ ]:
# chi_result.to_file("~/Documents/01_Research/larc-chicago-haq/chi-community-mean-test.json", driver="GeoJSON")

In [ ]:
#plot average no2 value
base = chicago.plot(color='white', edgecolor='black')

chi_result.plot(
    ax=base,
    figsize=(10,10),
    column='avg_value',
    cmap='viridis', 
    legend=True)

plt.title("Average NO2 Concentration ")

# plt.savefig("avg_no2.png")


In [ ]:
#plot number of stations per neighborhood
base = chicago.plot(color='white', edgecolor='black')

chi_result.plot(
    ax=base,
    figsize=(10,10),
    column='point_count',
    cmap='plasma', 
    legend=True)

plt.title("Station Count per Chicago Neighborhood")


In [ ]:
#calculate monthly average and variance for no2 
chi_summary_monthly = (
    chi_joined.groupby(["community", "NID", "month", "year", "month_year"])
    .agg(
        avg_value=("no2ConcIndividual.value", "mean"),
        var_value=("no2ConcIndividual.value", "var")
    )
    .reset_index()
)

#left join the above with the chicago shapefile data
chi_result_monthly = chicago.merge(chi_summary_monthly, on="NID", how="left")

chi_result_monthly

In [ ]:
#NOTE: These facet plots feel very hacky. If there is a better way to do it, then by all means.

#Facet plot the monthly NO2 averages
months = chi_result_monthly['month_year'].unique()
months = months[months.argsort()]

start_date = chi_result_monthly['month_year'].min()
end_date = chi_result_monthly['month_year'].max()

delta = relativedelta(months=1)

fig, axes = plt.subplots(
    nrows=3, ncols=4,
    figsize=(20, 15)
)

for ax, month in zip(axes.flatten(), range(1,13)):
# while start_date <= end_date:
    # ax = axes[i]

    # Base Chicago boundary
    chicago.plot(color='white', edgecolor='black', ax=ax)
    
    # Filter to month
    monthly_data = chi_result_monthly[chi_result_monthly['month_year'] == start_date]
    
    if not monthly_data.empty:
        monthly_data.plot(ax=ax, column='avg_value', cmap='BuPu',
                          vmin=chi_result_monthly['avg_value'].min(),
                          vmax=chi_result_monthly['avg_value'].max())
    
    # ax.set_title(calendar.month_name[month])
    # ax.axis('off')
    title_date = dt.date(start_date.year, start_date.month, start_date.day).strftime('%B %Y')
    ax.set_title(title_date, fontsize=12)
    ax.axis('off')

    start_date += delta

# Shared colorbar
sm = plt.cm.ScalarMappable(
    cmap='BuPu',
    norm=plt.Normalize(
        vmin=chi_result_monthly['avg_value'].min(),
        vmax=chi_result_monthly['avg_value'].max()
    )
)
sm._A = []
fig.colorbar(sm, ax=axes, shrink=0.5, label='Average NO2')

plt.suptitle('Monthly Avg NO2 - Chicago', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("month_avg_community.png")
plt.show()

In [ ]:

months = chi_result_monthly['month_year'].unique()
months = months[months.argsort()]

start_date = chi_result_monthly['month_year'].min()
end_date = chi_result_monthly['month_year'].max()

delta = relativedelta(months=1)

fig, axes = plt.subplots(
    nrows=3, ncols=4,
    figsize=(20, 15)
)

for ax, month in zip(axes.flatten(), range(1,13)):
    # ax = axes[i]
    
    # Base Chicago boundary
    chicago.plot(color='white', edgecolor='black', ax=ax)
    
    # Filter to month
    monthly_data = chi_result_monthly[chi_result_monthly['month_year'] == start_date]
    
    if not monthly_data.empty:
        monthly_data.plot(ax=ax, column='var_value', cmap='BuPu',
                          vmin=chi_result_monthly['var_value'].min(),
                          vmax=chi_result_monthly['var_value'].max())
    
    # ax.set_title(calendar.month_name[month])
    # ax.axis('off')
  
    title_date = dt.date(start_date.year, start_date.month, start_date.day).strftime('%B %Y')
    ax.set_title(title_date, fontsize=12)
    ax.axis('off')
    start_date += delta

# Shared colorbar
sm = plt.cm.ScalarMappable(
    cmap='BuPu',
    norm=plt.Normalize(
        vmin=chi_result_monthly['var_value'].min(),
        vmax=chi_result_monthly['var_value'].max()
    )
)
sm._A = []
fig.colorbar(sm, ax=axes, shrink=0.5, label='Average NO2')

plt.suptitle('Monthly NO2 Variance - Chicago', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("month_var_community.png")
plt.show()

In [ ]:
# Calculculate the percent of missing NO2 data per community per month.
chi_summary_na = (
    chi_joined.groupby(["community", "NID", "month", "month_year"])["no2ConcIndividual.value"]
    .apply(lambda x: x.isna().mean())
    .reset_index()
)

# Left joint with spatial dataset 
chi_result_na = chicago.merge(chi_summary_na, on="NID", how="left")

chi_result_na


In [ ]:


cook_clarity_na_count = 

months = chi_result_na['month_year'].unique()
months = months[months.argsort()]

start_date = chi_result_na['month_year'].min()
end_date = chi_result_na['month_year'].max()

delta = relativedelta(months=1)

fig, axes = plt.subplots(
    nrows=3, ncols=4,
    figsize=(20, 15)
)

for ax, month in zip(axes.flatten(), range(1,13)):
    # ax = axes[i]
    
    # Base Chicago boundary
    chicago.plot(color='white', edgecolor='black', ax=ax)
    
    # Filter to month
    monthly_data = chi_result_na[chi_result_na['month_year'] == start_date]
    
    if not monthly_data.empty:
        monthly_data.plot(ax=ax, column='no2ConcIndividual.value', cmap='BuPu',
                          vmin=chi_result_na['no2ConcIndividual.value'].min(),
                          vmax=chi_result_na['no2ConcIndividual.value'].max())
    
    # ax.set_title(calendar.month_name[month])
    # ax.axis('off')
  
    title_date = dt.date(start_date.year, start_date.month, start_date.day).strftime('%B %Y')
    ax.set_title(title_date, fontsize=12)
    ax.axis('off')
    start_date += delta

# Shared colorbar
sm = plt.cm.ScalarMappable(
    cmap='BuPu',
    norm=plt.Normalize(
        vmin=chi_result_na['no2ConcIndividual.value'].min(),
        vmax=chi_result_na['no2ConcIndividual.value'].max()
    )
)
sm._A = []
fig.colorbar(sm, ax=axes, shrink=0.5, label='%')

plt.suptitle('% Data Missing - Chicago', fontsize=16, y=1.02)
plt.tight_layout()
# plt.savefig("month_percent_missing-individual.png")
plt.show()

In [ ]:
#get % of missing per sensor
#note that this is just for the "no2ConcIndividual.value" variable in the dataframe
cook_clarity_na_count = cook_clarity.groupby(["longitude","latitude","month"])['no2ConcIndividual.value'].apply(lambda x: 100*x.isna().mean()).reset_index()

#Rename column
cook_clarity_na_count.rename(columns={"no2ConcIndividual.value": "num_missing_no2"}, inplace=True)

#Print first five rows
print(cook_clarity_na_count.head())

#Convert above dataframe to geopandas dataframe
#important to assign longitude and latitude for plotting
#This will need to be done for other statistics we calculate per station as well.
cook_clarity_na_count["geometry"] = gpd.points_from_xy(
    x=cook_clarity_na_count["longitude"],
    y=cook_clarity_na_count["latitude"]
)

#Set coordinate reference system to match crs of data we downloaded earlier
cook_clarity_na_count = gpd.GeoDataFrame(cook_clarity_na_count).set_crs(chicago.crs)
# Calculculate the percent of missing NO2 data per community per month.



In [ ]:

months = chi_result_na['month_year'].unique()
months = months[months.argsort()]

start_date = chi_result_na['month_year'].min()
end_date = chi_result_na['month_year'].max()

delta = relativedelta(months=1)

fig, axes = plt.subplots(
    nrows=3, ncols=4,
    figsize=(20, 15)
)

for ax, month in zip(axes.flatten(), range(1,13)):
    # ax = axes[i]
    
    # Base Chicago boundary
    chicago.plot(color='white', edgecolor='black', ax=ax)
    
    # Filter to month
    monthly_data = chi_result_na[chi_result_na['month_year'] == start_date]
    
    if not monthly_data.empty:
        monthly_data.plot(ax=ax, column='no2ConcIndividual.value', cmap='BuPu',
                          vmin=chi_result_na['no2ConcIndividual.value'].min(),
                          vmax=chi_result_na['no2ConcIndividual.value'].max())
    
    # ax.set_title(calendar.month_name[month])
    # ax.axis('off')
  
    title_date = dt.date(start_date.year, start_date.month, start_date.day).strftime('%B %Y')
    ax.set_title(title_date, fontsize=12)
    ax.axis('off')
    start_date += delta

# Shared colorbar
sm = plt.cm.ScalarMappable(
    cmap='BuPu',
    norm=plt.Normalize(
        vmin=chi_result_na['no2ConcIndividual.value'].min(),
        vmax=chi_result_na['no2ConcIndividual.value'].max()
    )
)
sm._A = []
fig.colorbar(sm, ax=axes, shrink=0.5, label='%')

plt.suptitle('% Data Missing - Chicago', fontsize=16, y=1.02)
plt.tight_layout()
plt.savefig("month_percent_missing.png")
plt.show()

In [ ]:

#get % of missing per sensor
#note that this is just for the "no2ConcIndividual.value" variable in the dataframe
# cook_clarity_na_count_month = (
#     cook_clarity.groupby(["sensor_name", "longitude", "latitude", "month"])["no2ConcIndividual.value"]
#     .apply(lambda x: 100 * x.isna().mean())
#     .reset_index()
# )

cook_clarity_na_count_month = (
    cook_clarity.groupby(["sensor_name", "longitude", "latitude","month"])["no2ConcIndividual.value"]
    .count()
    .reset_index(name="num_vals")
)

#Print first five rows
print(cook_clarity_na_count_month.head())

#Convert above dataframe to geopandas dataframe
#important to assign longitude and latitude for plotting
#This will need to be done for other statistics we calculate per station as well.
cook_clarity_na_count_month["geometry"] = gpd.points_from_xy(
    x=cook_clarity_na_count_month["longitude"],
    y=cook_clarity_na_count_month["latitude"]
)

#Set coordinate reference system to match crs of data we downloaded earlier
cook_clarity_na_count_month = gpd.GeoDataFrame(cook_clarity_na_count_month).set_crs(chicago.crs)

cook_clarity_na_count_month.to_csv("na_count_month.csv")